In [ ]:
// ==== Setup ====

import java.util.*;

interface MyListDouble {
    // appends e at the end
    void add(double e);

    // inserts e at pos, elements pos..size-1 move up by one
    // valid: 0 <= pos <= size
    void add(int pos, double e);

    // removes and returns the element at pos, elements pos+1..size-1 move down by one
    // valid: 0 <= pos < size
    double remove(int pos);

    // valid: 0 <= pos < size
    double get(int pos);

    // replaces the element at pos, returns the old one
    // valid: 0 <= pos < size
    double set(int pos, double e);

    // position of the first occurrence of e, or -1
    int indexOf(double e);

    int size();
    boolean isEmpty();
    void clear();
}

class Node {
    double data;
    Node next;          // null: no next node

    Node(double data) { this.data = data; }
}

void printAll(Node head) {
    Node current = head;
    while (current != null) {
        System.out.print(current.data + " ");
        current = current.next;
    }
    System.out.println();
}

# `MyLinkedListDouble`

## The complete class

In [ ]:
class MyLinkedListDouble implements MyListDouble {
    private static class Node {
        double data;
        Node next;

        Node(double data) { this.data = data; }
    }

    private Node head;      // first node, null if empty
    private Node tail;      // last node, null if empty
    private int size;

    public int size() { return size; }
    public boolean isEmpty() { return size == 0; }

    private void checkIndex(int pos) {
        if (pos < 0 || pos >= size)
            throw new IndexOutOfBoundsException("pos " + pos + ", size " + size);
    }

    // the node at pos, assumes 0 <= pos < size
    private Node nodeAt(int pos) {
        Node current = head;
        for (int i = 0; i < pos; i++)
            current = current.next;
        return current;
    }

    public double get(int pos) {
        checkIndex(pos);
        return nodeAt(pos).data;
    }

    public double set(int pos, double e) {
        checkIndex(pos);
        Node n = nodeAt(pos);
        double old = n.data;
        n.data = e;
        return old;
    }

    public int indexOf(double e) {
        int pos = 0;
        for (Node current = head; current != null; current = current.next) {
            if (current.data == e) return pos;
            pos++;
        }
        return -1;
    }

    public void addFirst(double e) {
        Node n = new Node(e);
        n.next = head;
        head = n;
        if (tail == null) tail = n;     // the list was empty
        size++;
    }

    public void add(double e) {
        Node n = new Node(e);
        if (tail == null) head = n;     // the list was empty
        else tail.next = n;
        tail = n;
        size++;
    }

    public void add(int pos, double e) {
        if (pos < 0 || pos > size)                  // pos == size is allowed
            throw new IndexOutOfBoundsException("pos " + pos + ", size " + size);
        if (pos == 0)    { addFirst(e); return; }
        if (pos == size) { add(e);      return; }
        Node before = nodeAt(pos - 1);
        Node n = new Node(e);
        n.next = before.next;
        before.next = n;
        size++;
    }

    public double removeFirst() {
        if (head == null) throw new NoSuchElementException("empty list");
        double removed = head.data;
        head = head.next;
        if (head == null) tail = null;  // removed the only node
        size--;
        return removed;
    }

    public double remove(int pos) {
        checkIndex(pos);
        if (pos == 0) return removeFirst();
        Node before = nodeAt(pos - 1);
        double removed = before.next.data;
        before.next = before.next.next;
        if (before.next == null) tail = before;     // removed the last node
        size--;
        return removed;
    }

    public void clear() {
        head = null;
        tail = null;
        size = 0;
    }
    @Override
    public String toString() {
        StringBuilder sb = new StringBuilder("[");     // avoids creating a new String at each step
        for (Node current = head; current != null; current = current.next) {
            sb.append(current.data);
            if (current.next != null) sb.append(", ");
        }
        return sb.append("]").toString();
    }
}

In [ ]:
MyLinkedListDouble nums = new MyLinkedListDouble();
for (int i = 1; i <= 5; i++) nums.add(i * 10);
System.out.println(nums);

nums.addFirst(5);               // front
nums.add(3, 25);                // middle
nums.add(nums.size(), 99);      // end
System.out.println(nums + "  size=" + nums.size());

System.out.println(nums.remove(0) + " " + nums.remove(2) + " " + nums.remove(nums.size() - 1) + " " + nums);
System.out.println(nums.get(2) + " " + nums.indexOf(40) + " " + nums.indexOf(7));

From one element to empty, and back:

In [ ]:
MyLinkedListDouble one = new MyLinkedListDouble();
one.add(1.5);
one.remove(0);
one.add(2.5);
one.add(3.5);
System.out.println(one + "  size=" + one.size());

## Cost: array vs. linked

| operation | array list | linked list |
|---|---|---|
| `get(i)`, `set(i, e)` | `O(1)` | `O(N)` |
| `add` front | `O(N)` | `O(1)` |
| `add` back | `O(1)` amortized | `O(1)` |
| `add(i, e)` middle | `O(N)` shift | `O(N)` walk |
| `remove` front | `O(N)` | `O(1)` |
| `remove` back | `O(1)` | `O(N)` |
| `remove(i)` middle | `O(N)` shift | `O(N)` walk |
| `indexOf` | `O(N)` | `O(N)` |
| `size`, `isEmpty`, `clear` | `O(1)` | `O(1)` |

`N` inserts at the front (wall-clock time, noisy: only the trend matters):

In [ ]:
long millis(Runnable r) {
    long start = System.nanoTime();
    r.run();
    return (System.nanoTime() - start) / 1_000_000;
}

System.out.printf("%8s %10s %10s%n", "N", "array ms", "linked ms");
for (int n = 20_000; n <= 80_000; n *= 2) {
    final int N = n;                        // lambdas need a final variable
    long arr = millis(() -> {
        List<Double> l = new ArrayList<>();
        for (int i = 0; i < N; i++) l.add(0, (double) i);
    });
    long lnk = millis(() -> {
        MyLinkedListDouble l = new MyLinkedListDouble();
        for (int i = 0; i < N; i++) l.addFirst(i);
    });
    System.out.printf("%,8d %10d %10d%n", N, arr, lnk);
}

- `remove` at the back is the only `O(N)` operation at the ends. What would fix it?
  - Keep a reference to the second-to-last node: cost of `removeLast` twice in a row?
- A program that uses `get(i)` heavily: which list?

# Iterating a linked list

## `get(i)` in a loop

```java
double sum = 0;
for (int i = 0; i < list.size(); i++)
    sum += list.get(i);
```

- Cost of `get(i)`? Of the whole loop?
- $0 + 1 + \dots + (N-1) = \frac{N(N-1)}{2}$ steps: `O(N²)` to read `N` elements
- Fix: an iterator that remembers a **node**, not an index

## Making the list `Iterable`

1. `implements Iterable<Double>`
2. an inner class implementing `Iterator<Double>`, holding the current node
3. `iterator()` returns a **new** instance of it

```java
private class Itr implements Iterator<Double> {
    private Node current = head;    // node holding the next element to return

    public boolean hasNext() { return current != null; }

    public Double next() {
        if (current == null) throw new NoSuchElementException();
        double e = current.data;
        current = current.next;
        return e;
    }
}

public Iterator<Double> iterator() { return new Itr(); }
```

- `Itr` is an **inner** class (no `static`): each `Itr` object belongs to one list object and reads its `head`.
- `Node` is `static`: a node does not need to know which list it belongs to.
- `hasNext`, `next`: `O(1)`

In [ ]:
class MyLinkedListDouble implements MyListDouble, Iterable<Double> {     // <===
    private static class Node {
        double data;
        Node next;

        Node(double data) { this.data = data; }
    }

    private Node head;      // first node, null if empty
    private Node tail;      // last node, null if empty
    private int size;

    public int size() { return size; }
    public boolean isEmpty() { return size == 0; }

    private void checkIndex(int pos) {
        if (pos < 0 || pos >= size)
            throw new IndexOutOfBoundsException("pos " + pos + ", size " + size);
    }

    // the node at pos, assumes 0 <= pos < size
    private Node nodeAt(int pos) {
        Node current = head;
        for (int i = 0; i < pos; i++)
            current = current.next;
        return current;
    }

    public double get(int pos) {
        checkIndex(pos);
        return nodeAt(pos).data;
    }

    public double set(int pos, double e) {
        checkIndex(pos);
        Node n = nodeAt(pos);
        double old = n.data;
        n.data = e;
        return old;
    }

    public int indexOf(double e) {
        int pos = 0;
        for (Node current = head; current != null; current = current.next) {
            if (current.data == e) return pos;
            pos++;
        }
        return -1;
    }

    public void addFirst(double e) {
        Node n = new Node(e);
        n.next = head;
        head = n;
        if (tail == null) tail = n;     // the list was empty
        size++;
    }

    public void add(double e) {
        Node n = new Node(e);
        if (tail == null) head = n;     // the list was empty
        else tail.next = n;
        tail = n;
        size++;
    }

    public void add(int pos, double e) {
        if (pos < 0 || pos > size)                  // pos == size is allowed
            throw new IndexOutOfBoundsException("pos " + pos + ", size " + size);
        if (pos == 0)    { addFirst(e); return; }
        if (pos == size) { add(e);      return; }
        Node before = nodeAt(pos - 1);
        Node n = new Node(e);
        n.next = before.next;
        before.next = n;
        size++;
    }

    public double removeFirst() {
        if (head == null) throw new NoSuchElementException("empty list");
        double removed = head.data;
        head = head.next;
        if (head == null) tail = null;  // removed the only node
        size--;
        return removed;
    }

    public double remove(int pos) {
        checkIndex(pos);
        if (pos == 0) return removeFirst();
        Node before = nodeAt(pos - 1);
        double removed = before.next.data;
        before.next = before.next.next;
        if (before.next == null) tail = before;     // removed the last node
        size--;
        return removed;
    }

    public void clear() {
        head = null;
        tail = null;
        size = 0;
    }

    private class Itr implements Iterator<Double> {             // <===
        private Node current = head;    // node holding the next element to return

        public boolean hasNext() { return current != null; }

        public Double next() {
            if (current == null) throw new NoSuchElementException();
            double e = current.data;
            current = current.next;
            return e;
        }
    }

    public Iterator<Double> iterator() { return new Itr(); }  // <===

    @Override
    public String toString() {
        StringBuilder sb = new StringBuilder("[");     // avoids creating a new String at each step
        for (Node current = head; current != null; current = current.next) {
            sb.append(current.data);
            if (current.next != null) sb.append(", ");
        }
        return sb.append("]").toString();
    }
}

In [ ]:
MyLinkedListDouble temps = new MyLinkedListDouble();
temps.add(21.5);
temps.add(19.0);
temps.add(23.25);
for (double t : temps)
    System.out.println(t);

### `get(i)` vs. iterator

In [ ]:
double sumWithGet(MyLinkedListDouble l) {
    double sum = 0;
    for (int i = 0; i < l.size(); i++) sum += l.get(i);
    return sum;
}

double sumWithIterator(MyLinkedListDouble l) {
    double sum = 0;
    for (double x : l) sum += x;
    return sum;
}

System.out.printf("%8s %10s %14s%n", "N", "get ms", "iterator ms");
for (int n = 5_000; n <= 40_000; n *= 2) {
    MyLinkedListDouble l = new MyLinkedListDouble();
    for (int i = 0; i < n; i++) l.add(i);
    System.out.printf("%,8d %10d %14d%n", n, millis(() -> sumWithGet(l)), millis(() -> sumWithIterator(l)));
}

`N` doubles: what happens to each column?

# Doubly linked lists

## Two links per node

```java
class Node {
    double data;
    Node next;
    Node prev;      // null for the first node
}
```

- Walk in both directions.
- Every change of structure now updates **both** directions.
- Memory: one more reference per node.

## Removing the last node

`tail.prev` is the node before the last one: no walk needed.

```java
public double removeLast() {
    if (tail == null) throw new NoSuchElementException("empty list");
    double removed = tail.data;
    tail = tail.prev;
    if (tail == null) head = null;  // removed the only node
    else tail.next = null;
    size--;
    return removed;
}
```

`O(1)`

## Removing a given node

With a reference to a node `n`, both neighbours are reachable: `n.prev`, `n.next`.

```java
private double unlink(Node n) {
    if (n.prev == null) head = n.next;      // n was the first node
    else n.prev.next = n.next;
    if (n.next == null) tail = n.prev;      // n was the last node
    else n.next.prev = n.prev;
    size--;
    return n.data;
}
```

- `O(1)`. Check the four cases: `n` first or not, `n` last or not.
- In a singly linked list, what do we need to remove a given node?

## Walking from the nearer end

```java
private Node nodeAt(int pos) {
    if (pos < size / 2) {
        Node current = head;
        for (int i = 0; i < pos; i++) current = current.next;
        return current;
    } else {
        Node current = tail;
        for (int i = size - 1; i > pos; i--) current = current.prev;
        return current;
    }
}
```

At most `N/2` steps. Cost?

## Inserting at a position

Four links change: two in the new node, one in each neighbour.

```java
Node after = nodeAt(pos);
Node before = after.prev;
Node n = new Node(e);
n.prev = before;
n.next = after;
before.next = n;
after.prev = n;
```

- `before` and `after` are saved first: does the order of the four assignments matter?
- Singly linked list: swapping the two link assignments broke the list. Why is this different?

## `MyDLinkedListDouble`

- Same interface as `MyLinkedListDouble`: `MyListDouble`, `Iterable<Double>`
- `removeFirst()`, `removeLast()`: `unlink(head)`, `unlink(tail)`
- `descendingIterator()`: from `tail`, following `prev`

In [ ]:
class MyDLinkedListDouble implements MyListDouble, Iterable<Double> {
    private static class Node {
        double data;
        Node next;
        Node prev;

        Node(double data) { this.data = data; }
    }

    private Node head;      // first node, null if empty
    private Node tail;      // last node, null if empty
    private int size;

    public int size() { return size; }
    public boolean isEmpty() { return size == 0; }

    private void checkIndex(int pos) {
        if (pos < 0 || pos >= size)
            throw new IndexOutOfBoundsException("pos " + pos + ", size " + size);
    }

    // the node at pos, assumes 0 <= pos < size
    private Node nodeAt(int pos) {
        if (pos < size / 2) {
            Node current = head;
            for (int i = 0; i < pos; i++) current = current.next;
            return current;
        } else {
            Node current = tail;
            for (int i = size - 1; i > pos; i--) current = current.prev;
            return current;
        }
    }

    public double get(int pos) {
        checkIndex(pos);
        return nodeAt(pos).data;
    }

    public double set(int pos, double e) {
        checkIndex(pos);
        Node n = nodeAt(pos);
        double old = n.data;
        n.data = e;
        return old;
    }

    public int indexOf(double e) {
        int pos = 0;
        for (Node current = head; current != null; current = current.next) {
            if (current.data == e) return pos;
            pos++;
        }
        return -1;
    }

    public void addFirst(double e) {
        Node n = new Node(e);
        n.next = head;
        if (head == null) tail = n;     // the list was empty
        else head.prev = n;
        head = n;
        size++;
    }

    public void add(double e) {
        Node n = new Node(e);
        n.prev = tail;
        if (tail == null) head = n;     // the list was empty
        else tail.next = n;
        tail = n;
        size++;
    }

    public void add(int pos, double e) {
        if (pos < 0 || pos > size)                  // pos == size is allowed
            throw new IndexOutOfBoundsException("pos " + pos + ", size " + size);
        if (pos == 0)    { addFirst(e); return; }
        if (pos == size) { add(e);      return; }
        Node after = nodeAt(pos);
        Node before = after.prev;
        Node n = new Node(e);
        n.prev = before;
        n.next = after;
        before.next = n;
        after.prev = n;
        size++;
    }

    private double unlink(Node n) {
        if (n.prev == null) head = n.next;      // n was the first node
        else n.prev.next = n.next;
        if (n.next == null) tail = n.prev;      // n was the last node
        else n.next.prev = n.prev;
        size--;
        return n.data;
    }

    public double remove(int pos) {
        checkIndex(pos);
        return unlink(nodeAt(pos));
    }

    public double removeFirst() {
        if (head == null) throw new NoSuchElementException("empty list");
        return unlink(head);
    }

    public double removeLast() {
        if (tail == null) throw new NoSuchElementException("empty list");
        return unlink(tail);
    }

    public void clear() {
        head = null;
        tail = null;
        size = 0;
    }

    private class Itr implements Iterator<Double> {
        private Node current = head;    // node holding the next element to return

        public boolean hasNext() { return current != null; }

        public Double next() {
            if (current == null) throw new NoSuchElementException();
            double e = current.data;
            current = current.next;
            return e;
        }
    }

    private class DescendingItr implements Iterator<Double> {
        private Node current = tail;

        public boolean hasNext() { return current != null; }

        public Double next() {
            if (current == null) throw new NoSuchElementException();
            double e = current.data;
            current = current.prev;
            return e;
        }
    }

    public Iterator<Double> iterator() { return new Itr(); }
    public Iterator<Double> descendingIterator() { return new DescendingItr(); }

    @Override
    public String toString() {
        StringBuilder sb = new StringBuilder("[");
        for (Node current = head; current != null; current = current.next) {
            sb.append(current.data);
            if (current.next != null) sb.append(", ");
        }
        return sb.append("]").toString();
    }

    // same list, following prev from tail: checks the prev links
    public String toStringBackward() {
        StringBuilder sb = new StringBuilder("[");
        for (Node current = tail; current != null; current = current.prev) {
            sb.append(current.data);
            if (current.prev != null) sb.append(", ");
        }
        return sb.append("]").toString();
    }
}

In [ ]:
MyDLinkedListDouble d = new MyDLinkedListDouble();
for (int i = 1; i <= 6; i++) d.add(i);
System.out.println(d + "  " + d.toStringBackward());

System.out.println(d.removeLast() + " " + d.remove(0) + " " + d.remove(2));
System.out.println(d + "  " + d.toStringBackward() + "  size=" + d.size());

d.addFirst(0);
d.add(2, 2.5);
System.out.println(d + "  " + d.toStringBackward() + "  size=" + d.size());

Iterator<Double> back = d.descendingIterator();
while (back.hasNext())
    System.out.print(back.next() + " ");
System.out.println();

# Circular lists

- Circular singly linked: `tail.next == head`
- Circular doubly linked: also `head.prev == tail`
- No `null` at the end: a traversal stops when it is back at the start

```java
if (head != null) {
    Node current = head;
    do {
        // use current.data
        current = current.next;
    } while (current != head);
}
```

- Why `do ... while` and not `while`?
- Circular singly linked, keeping **only** `tail`: where is the first node? Cost of adding at the front, adding at the back, removing from the front?
- Use: taking turns (round-robin), e.g. players in a game, processes sharing a CPU.

# `java.util.LinkedList`

## Inside `LinkedList`

From the JDK source:
- doubly linked, not circular
- fields: `first`, `last`, `size`
- `private static class Node<E> { E item; Node<E> next; Node<E> prev; }`
- `unlink(Node<E> x)`: same as ours
- `node(int index)`: walks from the nearer end
- implements `List<E>` and `Deque<E>` (double-ended queue): `addFirst`, `addLast`, `removeFirst`, `removeLast`, `getFirst`, `getLast`, `descendingIterator`, ...

In [ ]:
LinkedList<String> ll = new LinkedList<>(List.of("b", "c"));
ll.addFirst("a");
ll.addLast("d");
System.out.println(ll + " " + ll.getFirst() + " " + ll.getLast());
System.out.println(ll.removeFirst() + " " + ll.removeLast() + " " + ll);

## Removing through the iterator

- `it.remove()`: removes the element last returned by `it.next()`.
- `LinkedList`: the iterator holds the node, `unlink` is `O(1)`.
- `ArrayList`: the elements after it are shifted, `O(N)`.

In [ ]:
List<Integer> xs = new LinkedList<>();
for (int i = 0; i < 10; i++) xs.add(i);

Iterator<Integer> it = xs.iterator();
while (it.hasNext())
    if (it.next() % 2 == 0) it.remove();
System.out.println(xs);

Remove the even numbers from a list of `N` numbers: cost with `LinkedList`? With `ArrayList`?

In [ ]:
void removeEvens(List<Integer> l) {
    Iterator<Integer> it = l.iterator();
    while (it.hasNext())
        if (it.next() % 2 == 0) it.remove();
}

System.out.printf("%8s %10s %10s%n", "N", "array ms", "linked ms");
for (int n = 25_000; n <= 200_000; n *= 2) {
    List<Integer> arr = new ArrayList<>(), lnk = new LinkedList<>();
    for (int i = 0; i < n; i++) { arr.add(i); lnk.add(i); }
    System.out.printf("%,8d %10d %10d%n", n, millis(() -> removeEvens(arr)), millis(() -> removeEvens(lnk)));
}